In [2]:
import earthaccess
import xarray as xr
import rioxarray
print("所有库导入成功！")

所有库导入成功！


In [3]:
# persist=True 会把凭证存到 _netrc，下次就不用再登
auth = earthaccess.login(persist=True)

In [10]:
# 3. 搜索 2024年6-8月 的 MODIS Aqua L3 8天合成 9km 叶绿素数据
granules = earthaccess.search_data(
    short_name="MODISA_L3m_CHL",    # Aqua MODIS L3 叶绿素数据集
    granule_name="*.8D*.9km*",      # 过滤：8天合成 + 9km分辨率
    bounding_box=(117.8, 23.8, 118.5, 24.8),  # 厦门湾区域
    temporal=("2025-06-01", "2025-08-31")  # 时间范围
)
print(f"搜索到 {len(granules)} 个文件")
# 务必打印出每个文件的时间，确认数据连续性
for g in granules:
    print(g)

搜索到 26 个文件
Collection: {'ShortName': 'MODISA_L3m_CHL', 'Version': '2022.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'SouthBoundingCoordinate': -90, 'NorthBoundingCoordinate': 90, 'WestBoundingCoordinate': -180, 'EastBoundingCoordinate': 180}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-05-25T00:00:00Z', 'EndingDateTime': '2025-06-01T23:59:59Z'}}
Size(MB): 8.560454368591309
Data: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20250525_20250601.L3m.8D.CHL.chlor_a.9km.nc']
Collection: {'Version': '2022.0', 'ShortName': 'MODISA_L3m_CHL'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'NorthBoundingCoordinate': 90, 'WestBoundingCoordinate': -180, 'EastBoundingCoordinate': 180, 'SouthBoundingCoordinate': -90}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-05-25T00:00:00Z', 'EndingDateTime': '2025-06-01T23:59:59Z'}}
Size(MB): 8.85777378

In [15]:
# 去重：用字典保证 URL 唯一
unique_files = {}
for granule in granules:
    url = granule.data_links()[0]  # 取第一个链接
    if url not in unique_files:
        unique_files[url] = granule

print(f"去重前：{len(granules)} 个文件")
print(f"去重后：{len(unique_files)} 个文件")

# 转回列表方便后续操作
deduped_results = list(unique_files.values())

去重前：26 个文件
去重后：13 个文件


In [16]:
import re
from datetime import datetime

def parse_date_from_filename(url):
    """从 URL 中提取开始日期"""
    match = re.search(r'\.(\d{8})_(\d{8})\.', url)
    if match:
        start_str = match.group(1)
        return datetime.strptime(start_str, '%Y%m%d')
    return None

filtered = []
for granule in deduped_results:
    url = granule.data_links()[0]
    dt = parse_date_from_filename(url)
    if dt and dt >= datetime(2025, 6, 1) and dt <= datetime(2025, 8, 20):
        filtered.append(granule)

print(f"过滤后有效文件数：{len(filtered)}")

过滤后有效文件数：10


In [17]:
# 下载到本地文件夹（建议新建一个 data/ 目录）
downloaded_files = earthaccess.download(filtered, local_path="./data/")

print(f"成功下载 {len(downloaded_files)} 个文件")

f:\Users\E16\anaconda\Lib\site-packages\earthaccess\store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

成功下载 10 个文件


In [18]:
# 查看文件名
import glob
files = sorted(glob.glob("./data/*.nc"))
print(f"共 {len(files)} 个文件：")
for f in files:
    print(f.split("\\")[-1])  # Windows 下用 \\，Linux/Mac 用 /

共 10 个文件：
AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250610_20250617.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250618_20250625.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250626_20250703.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250704_20250711.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250712_20250719.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250720_20250727.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250728_20250804.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250805_20250812.L3m.8D.CHL.chlor_a.9km.nc
AQUA_MODIS.20250813_20250820.L3m.8D.CHL.chlor_a.9km.nc


In [21]:
import os
print("当前工作目录:", os.getcwd())

当前工作目录: d:\水产养殖2514\计算机\Python\.vscode\26-summer-modis-chla\part3_


In [22]:
import xarray as xr
import glob

# 使用原始字符串避免转义问题
data_dir = r"d:\水产养殖2514\计算机\Python\.vscode\26-summer-modis-chla\part3_\data"
file_list = sorted(glob.glob(data_dir + "/*.nc"))
print("找到的文件:", file_list[:3])  # 打印前三个确认

找到的文件: ['d:\\水产养殖2514\\计算机\\Python\\.vscode\\26-summer-modis-chla\\part3_\\data\\AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc', 'd:\\水产养殖2514\\计算机\\Python\\.vscode\\26-summer-modis-chla\\part3_\\data\\AQUA_MODIS.20250610_20250617.L3m.8D.CHL.chlor_a.9km.nc', 'd:\\水产养殖2514\\计算机\\Python\\.vscode\\26-summer-modis-chla\\part3_\\data\\AQUA_MODIS.20250618_20250625.L3m.8D.CHL.chlor_a.9km.nc']


In [24]:
import os
print(os.listdir("./data/"))   # 如果报错，说明这个目录不存在或不在当前位置

['AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250610_20250617.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250618_20250625.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250626_20250703.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250704_20250711.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250712_20250719.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250720_20250727.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250728_20250804.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250805_20250812.L3m.8D.CHL.chlor_a.9km.nc', 'AQUA_MODIS.20250813_20250820.L3m.8D.CHL.chlor_a.9km.nc']


# 检查一个文件的结构
file_path = r"d:\水产养殖2514\计算机\Python\.vscode\26-summer-modis-chla\part3_\data\AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc"
ds = xr.open_dataset(file_path)
print(ds)

In [5]:
import os
import glob

# 1. 确定当前工作目录
print("当前工作目录:", os.getcwd())

# 2. 用 glob 重新查找文件（确保在当前目录下找）
data_dir = "./data/"   # 或者改成你存放文件的目录
file_list = sorted(glob.glob(data_dir + "*.nc"))
print("找到的文件数量:", len(file_list))
if file_list:
    first_file = file_list[0]
    print("第一个文件路径:", first_file)
    print("路径是否存在?", os.path.exists(first_file))
else:
    print("没有找到任何 .nc 文件！请检查 data 文件夹的位置。")

当前工作目录: d:\Aquaculture2514\computer\Python\.vscode\26-summer-modis-chla\part3_
找到的文件数量: 10
第一个文件路径: ./data\AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc
路径是否存在? True


In [8]:
import xarray as xr
import glob
import os

# 1. 获取文件列表，并转换为绝对路径
data_dir = "./data/"
file_list = sorted(glob.glob(data_dir + "*.nc"))
file_list = [os.path.abspath(f) for f in file_list]  # 关键：转为绝对路径
print(f"共找到 {len(file_list)} 个文件")
print("第一个文件路径（绝对路径）:", file_list[0])

# 2. 检查第一个文件的结构
print("\n===== 第一个文件结构 =====")
ds_sample = xr.open_dataset(file_list[0])
print("=== 变量名 ===")
print(list(ds_sample.data_vars))

print("\n=== 坐标 ===")
print(list(ds_sample.coords))

print("\n=== 维度 ===")
print(dict(ds_sample.dims))


# 3. 关闭样本文件
ds_sample.close()

共找到 10 个文件
第一个文件路径（绝对路径）: d:\Aquaculture2514\computer\Python\.vscode\26-summer-modis-chla\part3_\data\AQUA_MODIS.20250602_20250609.L3m.8D.CHL.chlor_a.9km.nc

===== 第一个文件结构 =====
=== 变量名 ===
['chlor_a', 'palette']

=== 坐标 ===
['lat', 'lon']

=== 维度 ===
{'lat': 2160, 'lon': 4320, 'rgb': 3, 'eightbitcolor': 256}


C:\Users\E16\AppData\Local\Temp\ipykernel_29724\1516889243.py:22: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(dict(ds_sample.dims))


In [9]:
import numpy as np

# 打开第一个文件
ds_sample = xr.open_dataset(file_list[0])

# 查看 chlor_a 的基本统计
chl = ds_sample['chlor_a']
print(f"形状: {chl.shape}")
print(f"数据类型: {chl.dtype}")
print(f"有效值数量: {np.sum(~np.isnan(chl.values))}")  # 非NaN的数量
print(f"NaN数量: {np.sum(np.isnan(chl.values))}")      # NaN的数量

# 检查是否有 time 维度
print(f"\n是否有 time 维度: {'time' in ds_sample.dims}")
print(f"所有维度: {list(ds_sample.dims)}")

ds_sample.close()

形状: (2160, 4320)
数据类型: float32
有效值数量: 2613173
NaN数量: 6718027

是否有 time 维度: False
所有维度: ['lat', 'lon', 'rgb', 'eightbitcolor']
